# 📞 311 Public Feedback - Data Ingest
### Powered by Elasticsearch Serverless + Mistral

This notebook ingests **public feedback on 311 request/complaint types** - free-text messages New Yorkers submit suggesting new complaint categories - into Elasticsearch.

**What this notebook does:**
1. Fetches the feedback from [NYC Open Data](https://data.cityofnewyork.us/City-Government/Public-feedback-on-311-request-complaint-types/7ffd-6gs9) (dataset `7ffd-6gs9`) - no API key
2. Normalises each message (date, agency, status)
3. Bulk-indexes everything into `nyc_311_feedback`
4. **(Optional)** runs each message through **Mistral moderation** before you trust it

The free-text messages are ideal for Mistral: embeddings for semantic clustering, moderation for safety, and a chat model to summarise what the city is asking for.

---
> **Data source:** Public feedback on 311 request/complaint types, `7ffd-6gs9`, Socrata JSON API. This is a small, text-rich dataset. For a large, geo-tagged, time-series 311 set, see the "Go bigger" cell at the end.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests --quiet

In [ ]:
import requests
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Fetch the 311 feedback

In [ ]:
DATA_URL = 'https://data.cityofnewyork.us/resource/7ffd-6gs9.json'
resp = requests.get(DATA_URL, params={'$limit': 50000, '$order': 'feedback_date_time DESC'}, timeout=60)
resp.raise_for_status()
rows = resp.json()

print(f'✅ Fetched {len(rows)} feedback messages')
print()
for r in rows[:3]:
    print(f"  [{r.get('status')}] {r.get('agency_name')}: {r.get('customer_message','')[:80]}")

## 3. Create index with explicit mappings

In [ ]:
INDEX = 'nyc_311_feedback'

mapping = {
    'mappings': {
        'properties': {
            'feedback_date': {'type': 'date'},
            'message':       {'type': 'text'},
            'agency':        {'type': 'keyword'},
            'status':        {'type': 'keyword'},
        }
    }
}

if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
    print(f'🗑️  Deleted existing index: {INDEX}')

es.indices.create(index=INDEX, body=mapping)
print(f'✅ Created index: {INDEX}')

## 4. Enrich and ingest

In [ ]:
def enrich(r):
    doc = {
        'feedback_date': r.get('feedback_date_time'),
        'message':       r.get('customer_message'),
        'agency':        r.get('agency_name'),
        'status':        r.get('status'),
    }
    return {k: v for k, v in doc.items() if v not in (None, '')}

docs = [enrich(r) for r in rows if r.get('customer_message')]
actions = [{'_index': INDEX, '_source': d} for d in docs]

success, errors = helpers.bulk(es, actions, raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {success} messages ({len(errors)} errors)')

## 5. Verify

In [ ]:
resp = es.search(index=INDEX, size=0, body={
    'aggs': {
        'by_agency': {'terms': {'field': 'agency', 'size': 10}},
        'by_status': {'terms': {'field': 'status', 'size': 10}},
    }
})
print(f"Total messages: {resp['hits']['total']['value']}")
print('\nTop agencies:')
for b in resp['aggregations']['by_agency']['buckets']:
    print(f"  {b['key']:40s} {b['doc_count']}")
print('\nBy status:')
for b in resp['aggregations']['by_status']['buckets']:
    print(f"  {b['key']:20s} {b['doc_count']}")

## 6. (Optional) Moderate the feedback with Mistral 🛡️
Public free-text can contain toxic or unsafe content. Run each message through **Mistral's moderation API** and store the result in Elasticsearch so your agent can filter it.

> Needs your **Mistral API key**. Here we call Mistral's API directly (moderation isn't an Elasticsearch inference task type). See [mistral_guide.md](mistral_guide.md).

In [ ]:
MISTRAL_API_KEY = '<MISTRAL_API_KEY>'  # 🔑 paste your Mistral key

def moderate(text):
    r = requests.post(
        'https://api.mistral.ai/v1/moderations',
        headers={'Authorization': f'Bearer {MISTRAL_API_KEY}', 'Content-Type': 'application/json'},
        json={'model': 'mistral-moderation-latest', 'input': [text]},
        timeout=30,
    )
    r.raise_for_status()
    return r.json()['results'][0]['category_scores']

# Tag the first 10 messages with their highest moderation score, then update them in ES
sample = es.search(index=INDEX, size=10)['hits']['hits']
for h in sample:
    scores = moderate(h['_source']['message'])
    top_cat = max(scores, key=scores.get)
    es.update(index=INDEX, id=h['_id'],
              body={'doc': {'moderation_flag': top_cat, 'moderation_score': round(scores[top_cat], 3)}})
    print(f"  {scores[top_cat]:.2f} {top_cat:22s} | {h['_source']['message'][:60]}")
es.indices.refresh(index=INDEX)
print('✅ Tagged sample with Mistral moderation scores')

## 7. (Optional) Go bigger — the full 311 Service Requests dataset
`7ffd-6gs9` is small. For real aggregation projects (noise, heat, rats, by borough and time), swap in the giant **311 Service Requests** dataset `erm2-nwe9` (millions of rows, geo-tagged). Same Socrata pattern — just cap `$limit` and add a `$where` date filter.

In [ ]:
# Example: most recent 20k 311 service requests (geo + time + complaint_type)
BIG_URL = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'
big = requests.get(BIG_URL, params={
    '$limit': 20000,
    '$order': 'created_date DESC',
    '$select': 'created_date,complaint_type,descriptor,borough,latitude,longitude,agency_name,status',
}, timeout=120).json()
print(f'Fetched {len(big)} service requests')
print('Columns:', list(big[0].keys()))
# ... index into a `nyc_311_requests` index with the same bulk pattern as above,
#     mapping latitude/longitude into a geo_point and created_date as a date.

---
## ✅ Index loaded
`nyc_311_feedback` is ready. Add Mistral semantic search via [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md), or call Mistral directly for moderation, summaries, or an agent via [mistral_guide.md](mistral_guide.md).